# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

Question: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

Answer:

**Thrift schema (student.thrift)**

In [1]:
%%writefile student.thrift

struct Student {
    1: required string name,
    2: required i8 age,
    3: optional list<string> courses
}

service School {
    Student enrollCourse(1: required Student student, 2: required string course)
}

Writing student.thrift


**Thrift server (student_server.py)**

In [2]:
%%writefile student_thrift_server.py

import thriftpy2
student_thrift = thriftpy2.load("student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_server

class School(object):
    
    def enrollCourse(self, student, course): 
        student.courses.append(course)
        return student
    
server = make_server(student_thrift.School, School(), client_timeout=None)
print("Starting Server")
server.serve()

Writing student_thrift_server.py


Run the thrift server

**Thrift client (student_client.py)**

In [3]:
# Enter your student_client code here
import thriftpy2
student_thrift = thriftpy2.load("student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_client

school = make_client(student_thrift.School, timeout=None)

In [4]:
john_student = student_thrift.Student(
    name="John",
    age=30,
    courses=["Python"]
)

john_student = school.enrollCourse(john_student, "Data Engineering")

In [5]:
print(john_student)

Student(name='John', age=30, courses=['Python', 'Data Engineering'])


### Question 2

Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

Answer:

**Protobuf schema (book.proto)**

In [6]:
%%writefile book.proto
syntax = "proto3";

message Book {
    string title = 1;
    string author = 2;
    int32 page_count = 3;
}

service Library {
    rpc checkoutBook(Book) returns (Book) {}
}


Writing book.proto


Run

**Protobuf server (book_server.py)**

In [7]:
%%writefile book_server.py
from concurrent import futures
import grpc
import book_pb2_grpc

class Library(book_pb2_grpc.LibraryServicer):
    
    def checkoutBook(self, request, context):
        return request

server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
book_pb2_grpc.add_LibraryServicer_to_server(
    Library(), server)
server.add_insecure_port('[::]:50051')
print("Server started...")
print("Press Ctrl+C to stop the server.")
server.start()
server.wait_for_termination()

Writing book_server.py


Run

**Protobuf client (book_client.py)**

In [8]:
# Enter your book_client code here
import grpc
import book_pb2
import book_pb2_grpc

In [9]:
with grpc.insecure_channel('localhost:50051') as channel: 
    book = book_pb2.Book(
        title="Percy Jackson",
        author="Rick Riordan",
        page_count=420
    )
    
    stub = book_pb2_grpc.LibraryStub(channel)
    
    checked_out_book = stub.checkoutBook(book)
    
    print(checked_out_book)

title: "Percy Jackson"
author: "Rick Riordan"
page_count: 420

